# Czech UGS Storage Obligations — Monthly Regulatory Requirements

**Question this notebook answers**  
What volume of working gas must be held in Czech UGS at the **start of each heating-season month** (Oct–Mar) so that a 30-day peak-demand stress event (as defined in EU Regulation 2017/1938 Art. 6) can be survived, at each import-risk scenario (S1–S5)?

**Regulatory framing**  
Article 6 of the Gas Security of Supply Regulation requires that protected customers be supplied during:
- A **7-day extreme cold spell** at the 1-in-20 daily peak demand (`R.max.den`).
- A broader **30-day period** whose total consumption is given by `r_30dnu`.

The Czech implementation pins this 30-day stress period to calendar months, which is administratively tractable and conservative (the January peak is always captured by the January obligation).  The two-tier demand profile within each month is: 7 days at `R.max.den` followed by 23 days at the residual average implied by `r_30dnu`.

**Key design choices**  
- Monthly obligations are **independent stress tests** — each month asks "can we survive a 30-day event *starting this month*?" The 1-in-20 event occurs once per season, so months do not need to be linked by a carry-forward.
- Supplier compliance is checked at **month-start checkpoints** only (Oct 1, Nov 1, ..., Mar 1).  Intra-month draw-down paths are unconstrained, preserving market flexibility and respecting the incentive to withdraw during high-price events.
- The withdrawal-rate constraint is applied **within** each monthly simulation as fill depletes, because even a nominally sufficient volume cannot be delivered if fill drops below the rate-limiting zone of the capacity curve.
- **End-of-season reserve:** No endpoint constraint is embedded in the March stress test — doing so would implicitly provision for a second 1-in-20 event, inconsistent with the once-per-season premise.  A separate, modest operational floor of **~0.5 TWh** at end-March is recommended as a standalone policy instrument (see Section 8).


---
## 0. Setup

In [ ]:
%cd ..

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.interpolate import interp1d
from sklearn.isotonic import IsotonicRegression

import bsd.data as bdata
import bsd.jvs as jvs

jvs.apply_style(theme="light", context="notebook")
%matplotlib inline

DATA_FILE    = bdata.DATA_PATH_IMPORTS
PEAK_FILE    = bdata.DATA_PATH_DEMAND_PEAK
R30_FILE     = bdata.DATA_PATH_DEMAND_30DAY
STORAGE_FILE = bdata.DATA_PATH_STORAGE_GIE
CUTOFF       = bdata.DEFAULT_CUTOFF

CAPACITY_TWH  = 45.3034
CAPACITY_GWH  = CAPACITY_TWH * 1000

# Stress-period structure (EU Regulation Art. 6)
STRESS_PEAK_DAYS     = 7
STRESS_TOTAL_DAYS    = 30
STRESS_RESIDUAL_DAYS = STRESS_TOTAL_DAYS - STRESS_PEAK_DAYS  # 23

_sc = jvs.PALETTES["scenarios"]
SCENARIOS = {
    "S1": {"label": "S1 — High stress",   "percentile": 10, "color": _sc[0]},
    "S2": {"label": "S2 — Stressed",      "percentile": 20, "color": _sc[1]},
    "S3": {"label": "S3 — Base stressed", "percentile": 30, "color": _sc[2]},
    "S4": {"label": "S4 — Median",        "percentile": 50, "color": _sc[3]},
    "S5": {"label": "S5 — Favourable",    "percentile": 70, "color": _sc[4]},
}

MONTH_ORDER = [10, 11, 12, 1, 2, 3]
MONTH_NAMES = {10: "Oct", 11: "Nov", 12: "Dec", 1: "Jan", 2: "Feb", 3: "Mar"}
MONTH_NUM   = {"october": 10, "november": 11, "december": 12,
               "january": 1,  "february": 2,  "march": 3}
# Withdrawal curve assumption
# 'empirical'   — P95 isotonic fit from observed GIE data (conservative floor)
# 'engineering' — ENTSOG declared technical capacity with haircut (ceiling)
# 'blend'       — weighted average (default; recommended for regulatory use)
WC_ASSUMPTION       = "blend"
WC_HAIRCUT          = 0.10   # applied to engineering curve before blending
WC_BLEND_WEIGHT_ENG = 0.50   # weight on engineering curve (Branch 1: 50/50)

# Import capacity assumption
# 'scenarios'        — five S1–S5 percentile bands (default; risk axis)
# 'p99_cold'         — P99 of imports on cold days (top-COLD_DAY_QUANTILE withdrawal)
# 'p99_unconditional'— P99 of imports across all winter days
IMPORT_ASSUMPTION  = "p99_cold"   # controls which ceiling is shown in supplementary table
COLD_DAY_QUANTILE  = 0.80          # threshold defining a 'cold day'


---
## 1. Two-tier demand profile

The 30-day stress period has a two-tier demand structure:
- **Days 1–7**: peak daily demand `R.max.den` (the extreme cold-spell figure).
- **Days 8–30**: residual average = `(r_30dnu − 7 × R.max.den) / 23`.

This is lower than the peak because the 30-day total `r_30dnu` includes moderate days surrounding the extreme cold spell.

In [ ]:
peak_raw = pd.read_csv(PEAK_FILE)
peak_raw.columns = peak_raw.columns.str.strip()
peak_raw["month_num"] = peak_raw["month"].str.lower().map(MONTH_NUM)
peak_raw["peak_GWh_d"] = peak_raw["value [mhw]"] / 1000
peak_demand = peak_raw.set_index("month_num")["peak_GWh_d"].reindex(MONTH_ORDER)

r30_raw = pd.read_csv(R30_FILE)
r30_raw.columns = r30_raw.columns.str.strip()
r30_raw["month_num"] = r30_raw["month"].str.lower().map(MONTH_NUM)
r30_raw["total_GWh"] = r30_raw["value [mhw]"] / 1000
r30_total = r30_raw.set_index("month_num")["total_GWh"].reindex(MONTH_ORDER)

residual_demand = (r30_total - STRESS_PEAK_DAYS * peak_demand) / STRESS_RESIDUAL_DAYS

demand_profile = pd.DataFrame({
    "Peak 7d (GWh/d)":     peak_demand.round(1),
    "Residual 23d (GWh/d)": residual_demand.round(1),
    "Total 30d (GWh)":      r30_total.round(0),
}).rename(index=MONTH_NAMES)
demand_profile

---
## 2. Monthly import percentiles per scenario

In [ ]:
daily = bdata.load_daily_imports(DATA_FILE, cutoff=CUTOFF)
winter = daily[daily["month"].isin(MONTH_ORDER)].copy()

monthly_imports = {}
for key, sc in SCENARIOS.items():
    monthly_imports[key] = pd.Series(
        {m: winter[winter["month"] == m]["GWh_d"].quantile(sc["percentile"] / 100)
         for m in MONTH_ORDER}
    )

# Cold-day P99: identify top-COLD_DAY_QUANTILE withdrawal days per month
_sto = pd.read_csv(STORAGE_FILE, sep=";", parse_dates=["Gas Day Start (status at 6AM  CEST)"])
_sto.columns = _sto.columns.str.strip()
_sto = _sto.rename(columns={"Gas Day Start (status at 6AM  CEST)": "date",
                             "Withdrawal (GWh/d)": "withdrawal"})
_sto["date_key"] = pd.to_datetime(_sto["date"]).dt.normalize()
_w = winter.copy(); _w["date_key"] = _w["date"].dt.normalize()
_merged = _w.merge(_sto[["date_key", "withdrawal"]].dropna(), on="date_key", how="inner")

_cold_mask = pd.Series(False, index=_merged.index)
for _m in MONTH_ORDER:
    _mask = _merged["month"] == _m
    _thresh = _merged.loc[_mask, "withdrawal"].quantile(COLD_DAY_QUANTILE)
    _cold_mask = _cold_mask | (_mask & (_merged["withdrawal"] >= _thresh))

monthly_imports_p99_uncond = pd.Series(
    {m: _merged[_merged["month"] == m]["GWh_d"].quantile(0.99) for m in MONTH_ORDER}
)
monthly_imports_p99_cold = pd.Series(
    {m: _merged.loc[(_merged["month"] == m) & _cold_mask, "GWh_d"].quantile(0.99)
     for m in MONTH_ORDER}
)
IMPORT_CEILING = {"p99_cold": monthly_imports_p99_cold,
                  "p99_unconditional": monthly_imports_p99_uncond}[IMPORT_ASSUMPTION]

monthly_imports_df = pd.DataFrame(monthly_imports).round(1)
monthly_imports_df.index = [MONTH_NAMES[m] for m in MONTH_ORDER]

# Show scenario percentiles alongside P99 benchmarks
monthly_imports_df["P99 unconditional"] = monthly_imports_p99_uncond.values.round(1)
monthly_imports_df["P99 cold days"]     = monthly_imports_p99_cold.values.round(1)
print(f"Active ceiling assumption: {IMPORT_ASSUMPTION}")
monthly_imports_df

---
## 3. Withdrawal curve fit

Self-contained refit of the isotonic P95 relative capacity curve from `withdrawal_curve.ipynb`.  The curve maps fill level (%) to maximum deliverable withdrawal (GWh/d) and is used as a binding constraint within each monthly simulation as fill depletes.

In [ ]:
raw = pd.read_csv(STORAGE_FILE, sep=";",
                  parse_dates=["Gas Day Start (status at 6AM  CEST)"],
                  dayfirst=False, decimal=".")
raw.columns = raw.columns.str.strip()
raw = raw.rename(columns={
    "Gas Day Start (status at 6AM  CEST)": "date",
    "Full (%)":                            "fill_pct",
    "Withdrawal (GWh/d)":                  "withdrawal",
    "Withdrawal capacity (GWh/d)":         "wc_declared",
})
sto  = raw[["date", "fill_pct", "withdrawal", "wc_declared"]].dropna()
sto["month"] = sto["date"].dt.month
wint = sto[(sto["date"] >= CUTOFF) & sto["month"].isin(bdata.WINTER_MONTHS)].copy()
wint["util_ratio"] = wint["withdrawal"] / wint["wc_declared"]

WC_CURRENT = wint.loc[wint["date"] == wint["date"].max(), "wc_declared"].iloc[0]

BIN_EDGES  = np.arange(0, 105, 5)
BIN_LABELS = (BIN_EDGES[:-1] + 2.5).astype(float)
QUANTILE   = 0.95
MIN_OBS    = 5

wint["fill_bin"] = pd.cut(wint["fill_pct"], bins=BIN_EDGES,
                          labels=BIN_LABELS, right=False).astype(float)
bins = (wint.groupby("fill_bin", observed=False)
        .agg(n=("withdrawal", "count"),
             p95_abs=("withdrawal", lambda x: x.quantile(QUANTILE)),
             p95_ratio=("util_ratio", lambda x: x.quantile(QUANTILE)))
        .reset_index().rename(columns={"fill_bin": "fill_mid"}))
bins["p95_rel"] = bins["p95_ratio"] * WC_CURRENT
bins["reliable"] = bins["n"] >= MIN_OBS

reliable = bins[bins["reliable"]]
ir_rel = IsotonicRegression(increasing=True, out_of_bounds="clip")
ir_rel.fit(reliable["fill_mid"], reliable["p95_rel"])

fill_all = bins["fill_mid"].values
bins["curve_rel"] = ir_rel.predict(fill_all)

wc_curve_rel = interp1d(
    fill_all, bins["curve_rel"].values, kind="linear",
    bounds_error=False,
    fill_value=(bins["curve_rel"].iloc[0], bins["curve_rel"].iloc[-1]),
)

def wc(fill_pct):
    return float(wc_curve_rel(np.clip(fill_pct, 0, 100)))


# ── ENTSOG engineering curve (with haircut) ───────────────────────────────
_eng = pd.read_csv(Path("data/cz_usg_withdrawal_curve_2025.csv")).sort_values("percentile")
_eng_fill  = _eng["percentile"].values * 100
_eng_ratio = _eng["value"].values * (1 - WC_HAIRCUT)
_wc_engineering_interp = interp1d(
    _eng_fill, _eng_ratio * WC_CURRENT, kind="linear",
    bounds_error=False,
    fill_value=(_eng_ratio[0] * WC_CURRENT, _eng_ratio[-1] * WC_CURRENT),
)

def wc_engineering(fill_pct):
    return float(_wc_engineering_interp(np.clip(fill_pct, 0, 100)))

def wc_blend(fill_pct):
    w = WC_BLEND_WEIGHT_ENG
    return (1 - w) * wc_empirical(fill_pct) + w * wc_engineering(fill_pct)

wc_empirical = wc   # keep empirical accessible for sensitivity analysis
wc = {"empirical": wc_empirical, "engineering": wc_engineering, "blend": wc_blend}[WC_ASSUMPTION]

print(f"WC assumption: {WC_ASSUMPTION}  (haircut={WC_HAIRCUT:.0%}, eng_weight={WC_BLEND_WEIGHT_ENG:.0%})")
print(f"WC at 20%: {wc(20):.1f}  40%: {wc(40):.1f}  60%: {wc(60):.1f}  80%: {wc(80):.1f}  GWh/d")


---
## 4. Monthly simulation engine

Each month is simulated independently as a 30-day stress event starting at fill level `F`:

```
Days 1–7  : daily_gap = max(0, peak_demand[m] − import[m][s])
Days 8–30 : daily_gap = max(0, residual_demand[m] − import[m][s])

At each day t:
  1. max_wc_t = wc(fill_t)  ← withdrawal-rate constraint
  2. If max_wc_t < daily_gap_t → infeasible (rate-binding)
  3. fill_{t+1} = fill_t − daily_gap_t / (CAPACITY_GWH / 100)
  4. If fill_{t+1} < 0 → infeasible (volume-binding)
```

The minimum starting fill is found by bisection on `F ∈ [0%, 100%]`.  
No end-of-period constraint is applied: once the 30-day stress event is survived the obligation is met.

In [ ]:
def simulate_month(start_fill_pct, month, imp_GWh_d, *, wc_func=None):
    """Simulate the 30-day stress event for a single calendar month.

    Returns a dict: feasible, binding, day_of_infeasibility, fill_end, fill_trajectory.
    """
    if wc_func is None:
        wc_func = wc

    gap_peak  = max(0.0, float(peak_demand[month])    - imp_GWh_d)
    gap_resid = max(0.0, float(residual_demand[month]) - imp_GWh_d)
    gaps = [gap_peak] * STRESS_PEAK_DAYS + [gap_resid] * STRESS_RESIDUAL_DAYS

    fill = float(start_fill_pct)
    delta_per_gwh = 100.0 / CAPACITY_GWH  # % points per GWh withdrawn
    fill_traj = [fill]

    for t, g in enumerate(gaps):
        max_wc = wc_func(fill)
        if max_wc < g - 1e-9:
            return {"feasible": False, "binding": "withdrawal rate",
                    "day": t, "fill_end": fill, "fill_trajectory": fill_traj}
        fill -= g * delta_per_gwh
        if fill < -1e-9:
            return {"feasible": False, "binding": "volume",
                    "day": t, "fill_end": fill, "fill_trajectory": fill_traj}
        fill_traj.append(fill)

    return {"feasible": True, "binding": None,
            "day": None, "fill_end": fill, "fill_trajectory": fill_traj}


def min_start_fill_month(month, imp_GWh_d, *, tol=0.01, lo=0.0, hi=100.0, **sim_kwargs):
    """Bisection: minimum starting fill (%) for the month's stress event to be feasible."""
    if not simulate_month(hi, month, imp_GWh_d, **sim_kwargs)["feasible"]:
        return None  # infeasible even at 100% fill
    if simulate_month(lo, month, imp_GWh_d, **sim_kwargs)["feasible"]:
        return lo
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if simulate_month(mid, month, imp_GWh_d, **sim_kwargs)["feasible"]:
            hi = mid
        else:
            lo = mid
    return hi


# Quick smoke-test
r = simulate_month(50.0, 1, monthly_imports["S2"][1])
print(f"Jan S2 start=50%: feasible={r['feasible']}, fill_end={r['fill_end']:.1f}%")

---
## 5. Main results — minimum starting fill per month per scenario

In [ ]:
results = {}  # results[scenario_key][month] = {start_fill_pct, start_fill_TWh, binding, sim}

for key, sc in SCENARIOS.items():
    results[key] = {}
    for m in MONTH_ORDER:
        imp = float(monthly_imports[key][m])
        f   = min_start_fill_month(m, imp)
        sim = simulate_month(f if f is not None else 100.0, m, imp)
        results[key][m] = {
            "start_fill_pct": f,
            "start_fill_TWh": None if f is None else round(f * CAPACITY_TWH / 100, 2),
            "binding": sim["binding"] if f is None else (
                simulate_month(max(0.0, f - 0.05), m, imp)["binding"] or "volume"
            ),
            "fill_end": sim["fill_end"],
            "sim": sim,
        }

# Summary: TWh obligation table
oblig_twh = pd.DataFrame(
    {key: {MONTH_NAMES[m]: results[key][m]["start_fill_TWh"] for m in MONTH_ORDER}
     for key in SCENARIOS}
)
oblig_twh.columns = [SCENARIOS[k]["label"] for k in SCENARIOS]
print("Minimum required starting fill (TWh) per month per scenario")
oblig_twh

In [ ]:
# Fill-% version for regulatory context
oblig_pct = pd.DataFrame(
    {key: {MONTH_NAMES[m]: (
        None if results[key][m]["start_fill_pct"] is None
        else round(results[key][m]["start_fill_pct"], 1)
    ) for m in MONTH_ORDER}
     for key in SCENARIOS}
)
oblig_pct.columns = [SCENARIOS[k]["label"] for k in SCENARIOS]
print("Minimum required starting fill (% of capacity) per month per scenario")
oblig_pct

In [ ]:
# Binding constraint table
binding_table = pd.DataFrame(
    {key: {MONTH_NAMES[m]: results[key][m]["binding"] for m in MONTH_ORDER}
     for key in SCENARIOS}
)
binding_table.columns = [SCENARIOS[k]["label"] for k in SCENARIOS]
print("Binding constraint per month per scenario")
binding_table

---
## 6. Charts

In [ ]:
# Heatmap: obligation (TWh) by month × scenario
fig, ax = plt.subplots(figsize=(9, 3.5))
data_arr = oblig_twh.values.astype(float)
im = ax.imshow(data_arr.T, aspect="auto", cmap="YlOrRd", vmin=0, vmax=CAPACITY_TWH)
ax.set_xticks(range(len(MONTH_ORDER)))
ax.set_xticklabels([MONTH_NAMES[m] for m in MONTH_ORDER])
ax.set_yticks(range(len(SCENARIOS)))
ax.set_yticklabels([SCENARIOS[k]["label"] for k in SCENARIOS])
for i, m in enumerate(MONTH_ORDER):
    for j, key in enumerate(SCENARIOS):
        v = results[key][m]["start_fill_TWh"]
        txt = f"{v:.1f}" if v is not None else "—"
        ax.text(i, j, txt, ha="center", va="center", fontsize=8.5,
                color="white" if (v or 0) > CAPACITY_TWH * 0.55 else "#333333")
plt.colorbar(im, ax=ax, label="TWh", fraction=0.03)
ax.set_title("Monthly storage obligation (TWh) — minimum fill at month start")
fig.tight_layout()
fig.savefig("figs/storage_obligations_heatmap.png", bbox_inches="tight")
fig;

In [ ]:
# Grouped bar: all months × scenarios
fig, ax = plt.subplots(figsize=(11, 4.5))
n_months = len(MONTH_ORDER)
n_scen   = len(SCENARIOS)
width    = 0.14
xs       = np.arange(n_months)

for j, (key, sc) in enumerate(SCENARIOS.items()):
    vals = [results[key][m]["start_fill_TWh"] or 0.0 for m in MONTH_ORDER]
    offset = (j - n_scen / 2 + 0.5) * width
    ax.bar(xs + offset, vals, width=width, color=sc["color"], alpha=0.85, label=sc["label"])

ax.axhline(CAPACITY_TWH, color="black", lw=1.0, ls="--",
           label=f"Total capacity ({CAPACITY_TWH:.1f} TWh)")
ax.set_xticks(xs)
ax.set_xticklabels([MONTH_NAMES[m] for m in MONTH_ORDER])
ax.set_ylabel("Minimum required fill (TWh)")
ax.set_title("Monthly storage obligations by scenario")
ax.legend(fontsize=8, ncol=3)
ax.set_ylim(0, CAPACITY_TWH * 1.08)
jvs.grid(ax=ax)
fig.tight_layout()
fig.savefig("figs/storage_obligations_bar.png", bbox_inches="tight")
fig;

In [ ]:
# Fill trajectories for January (binding month) under each scenario
fig, ax = plt.subplots(figsize=(8, 4))
days = np.arange(STRESS_TOTAL_DAYS + 1)
day_labels = ["Jan 1", "", "", "", "", "", "", "Jan 8\n(residual)",
              *[""] * 21, "Jan 31"]

for key, sc in SCENARIOS.items():
    info = results[key][1]  # January
    traj = info["sim"]["fill_trajectory"]
    f    = info["start_fill_pct"]
    ax.plot(range(len(traj)), traj, color=sc["color"], lw=2,
            label=f"{sc['label']}  start={f:.1f}%" if f is not None else sc["label"])

ax.axvline(STRESS_PEAK_DAYS, color="grey", lw=0.8, ls=":", alpha=0.7)
ax.text(STRESS_PEAK_DAYS + 0.3, ax.get_ylim()[1] * 0.97, "peak → residual",
        fontsize=7.5, color="grey", va="top")
ax.axhline(20, color="grey", lw=0.6, ls="--", alpha=0.6,
           label="Low-confidence WC zone (<20%)")
ax.set_xlabel("Day of stress period")
ax.set_ylabel("Fill level (%)")
ax.set_title("January: fill trajectory over 30-day stress period (from minimum required start)")
ax.legend(fontsize=8)
jvs.grid(ax=ax)
fig.tight_layout()
fig.savefig("figs/storage_obligations_jan_trajectories.png", bbox_inches="tight")
fig;

---
## 7. Regulatory summary table

The table below is the primary regulatory output: minimum working-gas volume that each regulated supplier must hold at the start of each month, expressed in TWh (to be allocated to individual suppliers in proportion to their share of protected customers).

S2 (P20 imports × 1-in-20 demand ≈ 1-in-100 joint event) is the **recommended planning anchor**.

In [ ]:
# Formatted summary: TWh with binding constraint annotations
rows = []
for m in MONTH_ORDER:
    row = {"Month": MONTH_NAMES[m]}
    for key, sc in SCENARIOS.items():
        info = results[key][m]
        v = info["start_fill_TWh"]
        b = info["binding"]
        if v is None:
            row[sc["label"]] = "infeasible"
        elif v == 0.0:
            row[sc["label"]] = "0.00"
        else:
            marker = " ★" if b == "withdrawal rate" else ""
            row[sc["label"]] = f"{v:.2f}{marker}"
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index("Month")
print("★ withdrawal-rate binding  (blank = volume binding)")
print(f"Czech UGS total capacity: {CAPACITY_TWH} TWh")
print("Recommended end-of-season operational floor (separate instrument): ~0.5 TWh at Mar 31\n")
summary_df

---
## 8. Caveats

**1. Monthly independence assumption.**  Each month's obligation is sized for a standalone 30-day stress event.  The 1-in-20 demand event occurs once per season, so no carry-forward between months is needed.  Physical fill levels entering each month depend on actual draw-down earlier in the season; injection-season feasibility (Apr–Sep) is not checked here.

**2. Deterministic imports.**  Month-specific percentiles are applied as constants throughout each 30-day period.  Within-month import variability would allow partial buffering across days, so results are conservative upper-bounds at each scenario.

**3. Withdrawal curve at low fill levels.**  The curve is poorly identified below ~20% fill (few observations in that regime under winter stress).  The flat extrapolation used may understate true capacity there; any obligation whose trajectory passes below 20% should be treated with additional caution.

**4. Demand/import correlation — commercially driven, not a model bias.**  
Post-2022 Physical Flow data shows a strong negative correlation between Czech daily demand and imports (Spearman ρ ≈ −0.52, p < 0.001).  The mechanism is commercial: on cold days, German hub prices spike, Czech-German locational spreads widen, and suppliers find it cheaper to withdraw from domestic storage (sunk cost) than to import at elevated hub prices.  Physical import capacity is *not* similarly constrained — the pipe could carry more.

Consequence: the unconditional import percentiles used in this model are *conservative*.  They include days when imports were commercially suppressed by high prices; in a genuine Article 13 security-of-supply emergency, regulators can mandate or financially incentivise imports above those levels.  The model does not overstate obligations because of this correlation; if anything it is slightly too conservative.

The residual risk that *is* not captured is a **pan-European physical supply crisis** — a prolonged cold spell saturating German import capacity system-wide, where commercial override is impossible.  That scenario is closer to S1 (P10 imports) than S2.  S1 should be interpreted as the physical-infrastructure stress case; S2 as the conservative commercial-conditions anchor.

**5. Czech monthly framing vs. EU floating window.**  The EU regulation's 7/30-day stress events are not pinned to calendar months — they can start on any day.  The Czech monthly approach is conservative (January always captures the worst peak) and administratively tractable.  A cross-month stress event (e.g. starting January 15) is not explicitly modelled; at the margin this could require modestly higher obligations for boundary months.

**6. End-of-season operational floor (separate instrument).**  The March stress test carries no endpoint constraint — imposing one would implicitly provision for a second 1-in-20 event.  However, a modest end-of-season reserve is prudent to guard against a brief April cold snap before injection season is underway.  A floor of **~0.5 TWh at March 31** (covering roughly 5 days of the S2 March peak gap at 49 GWh/d) is recommended as a standalone regulatory instrument, distinct from and additional to the monthly stress-test obligations above.
